# Setup

## MySql connection Test

In [26]:
! pip install python-dotenv
! pip install haversine==2.8.1 mysql-connector-python==8.0.33 tabulate==0.9.0
! pip install mysql-connector-python pandas ipykernel
! pip install folium


   ----- ---------------------------------- 1/7 [urllib3]
   ----- ---------------------------------- 1/7 [urllib3]
   ----- ---------------------------------- 1/7 [urllib3]
   ----- ---------------------------------- 1/7 [urllib3]
   ----------- ---------------------------- 2/7 [charset_normalizer]
   ----------- ---------------------------- 2/7 [charset_normalizer]
   ---------------------- ----------------- 4/7 [requests]
   ---------------------- ----------------- 4/7 [requests]
   ---------------------------- ----------- 5/7 [branca]
   ---------------------------------- ----- 6/7 [folium]
   ---------------------------------- ----- 6/7 [folium]
   ---------------------------------- ----- 6/7 [folium]
   ---------------------------------- ----- 6/7 [folium]
   ---------------------------------------- 7/7 [folium]



In [28]:
import mysql.connector as mysql
from tabulate import tabulate

class DbConnector:
    """
    Connects to the MySQL server on the Ubuntu virtual machine.
    """

    def __init__(self,
                 HOST="tdt4225-03.idi.ntnu.no",
                 DATABASE="db_A2_G3",
                 USER="DiegoT",
                 PASSWORD="G3Rocky"):
        self.db_connection = None
        self.cursor = None

        try:
            self.db_connection = mysql.connect(
                host=HOST,
                database=DATABASE,
                user=USER,
                password=PASSWORD,
                port=3306
            )
            self.cursor = self.db_connection.cursor()

            print("Connected to:", self.db_connection.get_server_info())
            self.cursor.execute("SELECT DATABASE();")
            database_name = self.cursor.fetchone()
            print("You are connected to the database:", database_name)
            print("-----------------------------------------------\n")

        except mysql.Error as e:
            print("ERROR: Failed to connect to db:", e)
            raise e  # Re-raise so main() handles setup failure cleanly

    def close_connection(self):
        if self.cursor:
            self.cursor.close()
        if self.db_connection and self.db_connection.is_connected():
            print("\n-----------------------------------------------")
            print("Connection to %s is closed" % self.db_connection.get_server_info())
            self.db_connection.close()


class ExampleProgram:

    def __init__(self):
        self.connection = DbConnector()
        self.db_connection = self.connection.db_connection
        self.cursor = self.connection.cursor

    def create_table(self, table_name):
        # Safely formatted SQL query template
        query = f"""CREATE TABLE IF NOT EXISTS {table_name} (
                   id INT AUTO_INCREMENT NOT NULL PRIMARY KEY,
                   name VARCHAR(30))"""
        self.cursor.execute(query)
        self.db_connection.commit()

    def insert_data(self, table_name):
        names = ['Bobby', 'Mc', 'McSmack', 'Board']
        # Recommended: Use parameterized queries to avoid syntax errors/SQL injection
        query = f"INSERT INTO {table_name} (name) VALUES (%s)"
        for name in names:
            self.cursor.execute(query, (name,))
        self.db_connection.commit()

    def fetch_data(self, table_name):
        query = f"SELECT * FROM {table_name}"
        self.cursor.execute(query)
        rows = self.cursor.fetchall()
        print("Data from table %s, raw format:" % table_name)
        print(rows)
        print("Data from table %s, tabulated:" % table_name)
        print(tabulate(rows, headers=self.cursor.column_names))
        return rows

    def drop_table(self, table_name):
        print("Dropping table %s..." % table_name)
        query = f"DROP TABLE {table_name}"
        self.cursor.execute(query)

    def show_tables(self):
        self.cursor.execute("SHOW TABLES")
        rows = self.cursor.fetchall()
        print(tabulate(rows, headers=self.cursor.column_names))


def main():
    program = None
    try:
        program = ExampleProgram()
        program.create_table(table_name="Person")
        program.insert_data(table_name="Person")
        _ = program.fetch_data(table_name="Person")
        program.drop_table(table_name="Person")
        program.show_tables()
    except Exception as e:
        print("ERROR: Failed to use database:", e)
    finally:
        if program and hasattr(program, 'connection'):
            program.connection.close_connection()


if __name__ == '__main__':
    main()

Connected to: 8.4.11-0ubuntu0.26.04.1
You are connected to the database: ('db_A2_G3',)
-----------------------------------------------

Data from table Person, raw format:
[(1, 'Bobby'), (2, 'Mc'), (3, 'McSmack'), (4, 'Board')]
Data from table Person, tabulated:
  id  name
----  -------
   1  Bobby
   2  Mc
   3  McSmack
   4  Board
Dropping table Person...
Tables_in_db_A2_G3
--------------------

-----------------------------------------------
Connection to 8.4.11-0ubuntu0.26.04.1 is closed


## Df load

In [13]:
DATA_PATH = "porto.csv"

In [14]:
import pandas as pd
import numpy as np

# ---- Load the dataframe ----
# Keep IDs-with-NULLs as nullable ints, and keep POLYLINE as a raw string for now
# (parsing 1.7M JSON lists is expensive, so we do it lazily / when needed).

dtypes = {
    "TRIP_ID": "Int64",
    "CALL_TYPE": "category",
    "ORIGIN_CALL": "Int64",
    "ORIGIN_STAND": "Int64",
    "TAXI_ID": "Int64",
    "TIMESTAMP": "Int64",
    "DAYTYPE": "category",
    "MISSING_DATA": "string",     # normalised to bool in the next cell
    "POLYLINE": "string",
}

df = pd.read_csv(DATA_PATH, dtype=dtypes)
df.columns = [c.strip().upper() for c in df.columns]   # tolerate header case/whitespace

# Part 1 EDA

In [15]:
# df copied to df_clean for cleaning and analysis, so we can keep the original df intact
df_clean = df.copy()

print(df_clean.shape)
df_clean.head()

(1710670, 9)


,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,TIMESTAMP,DAY_TYPE,MISSING_DATA,POLYLINE
0,1372636858620000589,C,<NA>,<NA>,20000589,1372636858,A,False,"[[-8.618643,41.141412],[-8.618499,41.141376],[..."
1,1372637303620000596,B,<NA>,7,20000596,1372637303,A,False,"[[-8.639847,41.159826],[-8.640351,41.159871],[..."
2,1372636951620000320,C,<NA>,<NA>,20000320,1372636951,A,False,"[[-8.612964,41.140359],[-8.613378,41.14035],[-..."
3,1372636854620000520,C,<NA>,<NA>,20000520,1372636854,A,False,"[[-8.574678,41.151951],[-8.574705,41.151942],[..."
4,1372637091620000337,C,<NA>,<NA>,20000337,1372637091,A,False,"[[-8.645994,41.18049],[-8.645949,41.180517],[-..."


In [16]:
# ---- Basic cleaning ----
df_clean.info()

# MISSING_DATA -> real boolean
df_clean["MISSING_DATA"] = (
    df_clean["MISSING_DATA"].str.strip().str.lower().map({"true": True, "false": False})
).astype("boolean")

# TIMESTAMP (unix seconds) -> datetime
df_clean["START_TIME"] = pd.to_datetime(df_clean["TIMESTAMP"], unit="s")
df_clean.drop(columns=["TIMESTAMP"], inplace=True)

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1710670 entries, 0 to 1710669
Data columns (total 9 columns):
 #   Column        Dtype   
---  ------        -----   
 0   TRIP_ID       Int64   
 1   CALL_TYPE     category
 2   ORIGIN_CALL   Int64   
 3   ORIGIN_STAND  Int64   
 4   TAXI_ID       Int64   
 5   TIMESTAMP     Int64   
 6   DAY_TYPE      object  
 7   MISSING_DATA  string  
 8   POLYLINE      string  
dtypes: Int64(5), category(1), object(1), string(2)
memory usage: 114.2+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1710670 entries, 0 to 1710669
Data columns (total 9 columns):
 #   Column        Dtype        
---  ------        -----        
 0   TRIP_ID       Int64        
 1   CALL_TYPE     category     
 2   ORIGIN_CALL   Int64        
 3   ORIGIN_STAND  Int64        
 4   TAXI_ID       Int64        
 5   DAY_TYPE      object       
 6   MISSING_DATA  boolean      
 7   POLYLINE      string       
 8   START_TIME    datetime64[s]
dtypes: Int64(4), boolean(1),

In [17]:
df_clean.describe(include='all')

,TRIP_ID,CALL_TYPE,ORIGIN_CALL,ORIGIN_STAND,TAXI_ID,DAY_TYPE,MISSING_DATA,POLYLINE,START_TIME
count,1710670.0,1710670,364770.0,806579.0,1710670.0,1710670,1710670,1710670,1710670
unique,<NA>,3,<NA>,<NA>,<NA>,1,2,1703650,NaN
top,<NA>,B,<NA>,<NA>,<NA>,A,False,[],NaN
freq,<NA>,817881,<NA>,<NA>,<NA>,1710670,1710660,5901,NaN
mean,1388622387803249152.0,NaN,24490.363018,30.272381,20000348.531181,NaN,NaN,NaN,2014-01-02 00:26:27
min,1372636853620000256.0,NaN,2001.0,1.0,20000001.0,NaN,NaN,NaN,2013-07-01 00:00:53
25%,1380730539120000256.0,NaN,6593.0,15.0,20000166.0,NaN,NaN,NaN,2013-10-02 16:15:38
50%,1388492544120000000.0,NaN,18755.0,27.0,20000345.0,NaN,NaN,NaN,2013-12-31 12:22:23
75%,1396750457120000256.0,NaN,40808.0,49.0,20000520.0,NaN,NaN,NaN,2014-04-06 02:14:16
max,1404172796620000768.0,NaN,63884.0,63.0,20000981.0,NaN,NaN,NaN,2014-06-30 23:59:56


In [6]:
print("Rows:", len(df_clean))
print("Unique taxis:", df_clean["TAXI_ID"].nunique())
print("Unique trip ids:", df_clean["TRIP_ID"].nunique())
print("\nNulls per column:\n", df_clean.isna().sum())
print("\nCALL_TYPE:\n", df_clean["CALL_TYPE"].value_counts(dropna=False))
print("\nDAYTYPE:\n", df_clean["DAY_TYPE"].value_counts(dropna=False))
print("\nMISSING_DATA:\n", df_clean["MISSING_DATA"].value_counts(dropna=False))

Rows: 1710670
Unique taxis: 448
Unique trip ids: 1710589

Nulls per column:
 TRIP_ID               0
CALL_TYPE             0
ORIGIN_CALL     1345900
ORIGIN_STAND     904091
TAXI_ID               0
DAY_TYPE              0
MISSING_DATA          0
POLYLINE              0
START_TIME            0
dtype: int64

CALL_TYPE:
 CALL_TYPE
B    817881
C    528019
A    364770
Name: count, dtype: int64

DAYTYPE:
 DAY_TYPE
A    1710670
Name: count, dtype: int64

MISSING_DATA:
 MISSING_DATA
False    1710660
True          10
Name: count, dtype: Int64


In [7]:
# Duplicate Trips
print("Number of duplicate trips:", df_clean.duplicated(subset=["TRIP_ID"]).sum())

# Show duplicate trip 
print("\n",df_clean[df_clean.duplicated(subset=["TRIP_ID"], keep=False)].sort_values(by=["TRIP_ID", "START_TIME"]))

# Same trip id but different polyline ?
print("\nNumber of duplicate trips:", df_clean.duplicated(subset=["TRIP_ID","POLYLINE"]).sum())

Number of duplicate trips: 81

                      TRIP_ID CALL_TYPE  ORIGIN_CALL  ORIGIN_STAND   TAXI_ID  \
3519     1372702836620000080         B         <NA>            16  20000080   
3720     1372702836620000080         C         <NA>          <NA>  20000080   
21282    1373025987620000601         A         2002          <NA>  20000601   
21385    1373025987620000601         C         <NA>          <NA>  20000601   
33228    1373210896620000598         B         <NA>            57  20000598   
...                      ...       ...          ...           ...       ...   
1665302  1403407180620000242         C         <NA>          <NA>  20000242   
1684526  1403716904620000337         C         <NA>          <NA>  20000337   
1684739  1403716904620000337         C         <NA>          <NA>  20000337   
1705646  1404093316620000307         B         <NA>            57  20000307   
1705665  1404093316620000307         C         <NA>          <NA>  20000307   

        DAY_TYPE  M

In [18]:
""" 
Verify origin call and stand statements:
ORIGIN_CALL: ID of the client who initiated the call (only set when CALL_TYPE = A). Otherwise NULL.
ORIGIN_STAND: Taxi stand ID where the trip started (only set when CALL_TYPE = B). Otherwise NULL.
"""

# 1. Validation Logic
origin_call_valid = (
    df_clean[df_clean['CALL_TYPE'] == 'A']['ORIGIN_CALL'].notna().all() and
    df_clean[df_clean['CALL_TYPE'] != 'A']['ORIGIN_CALL'].isna().all()
)

origin_stand_valid = (
    df_clean[df_clean['CALL_TYPE'] == 'B']['ORIGIN_STAND'].notna().all() and
    df_clean[df_clean['CALL_TYPE'] != 'B']['ORIGIN_STAND'].isna().all()
)

print(f"ORIGIN_CALL condition valid: {origin_call_valid}")
print(f"ORIGIN_STAND condition valid: {origin_stand_valid}\n")

# 2. Detailed Breakdown Table (observed=False silences the FutureWarning)
summary = df_clean.groupby('CALL_TYPE', observed=False).agg(
    TOTAL_TRIPS=('CALL_TYPE', 'count'),
    ORIGIN_CALL_NON_NULL=('ORIGIN_CALL', 'count'),
    ORIGIN_STAND_NON_NULL=('ORIGIN_STAND', 'count')
)

# Calculate missing stand IDs for CALL_TYPE 'B'
summary['ORIGIN_STAND_MISSING'] = summary.apply(
    lambda row: row['TOTAL_TRIPS'] - row['ORIGIN_STAND_NON_NULL'] if row.name == 'B' else 0, 
    axis=1
)

print("--- Data Completeness Breakdown ---")
print(summary)

ORIGIN_CALL condition valid: True
ORIGIN_STAND condition valid: False

--- Data Completeness Breakdown ---
           TOTAL_TRIPS  ORIGIN_CALL_NON_NULL  ORIGIN_STAND_NON_NULL  \
CALL_TYPE                                                             
A               364770                364770                      0   
B               817881                     0                 806579   
C               528019                     0                      0   

           ORIGIN_STAND_MISSING  
CALL_TYPE                        
A                             0  
B                         11302  
C                             0  


In [19]:
# 1. Calculate polyline point length to identify the record with better GPS data
df_clean['POLYLINE_LEN'] = df_clean['POLYLINE'].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

# 2. Sort by TRIP_ID and POLYLINE_LEN (descending) so complete trajectories come first
df_clean = df_clean.sort_values(
    by=['TRIP_ID', 'POLYLINE_LEN'], 
    ascending=[True, False]
)

# 3. Drop duplicate TRIP_IDs, retaining the first row (the one with the longest POLYLINE)
df_clean = df_clean.drop_duplicates(subset=['TRIP_ID'], keep='first')

# 4. Clean up temporary column
df_clean = df_clean.drop(columns=['POLYLINE_LEN'])

# Verify duplicates are completely eliminated
print("Remaining duplicate TRIP_IDs:", df_clean.duplicated(subset=['TRIP_ID']).sum())

Remaining duplicate TRIP_IDs: 0


In [20]:
# Origin stand and taxi id sample
df_clean[df_clean['CALL_TYPE'] == 'B'][['ORIGIN_STAND', 'TAXI_ID', 'POLYLINE']].sample(10, random_state=42)

,ORIGIN_STAND,TAXI_ID,POLYLINE
1527074,11,20000663,"[[-8.613324,41.166891],[-8.613756,41.166927],[..."
740124,23,20000506,"[[-8.612478,41.145876],[-8.612478,41.145858],[..."
1444895,15,20000273,"[[-8.585928,41.148513],[-8.585811,41.148612],[..."
1632332,57,20000625,"[[-8.610786,41.145714],[-8.610687,41.146002],[..."
805034,27,20000515,"[[-8.608779,41.147721],[-8.608779,41.147748],[..."
586798,51,20000565,"[[-8.6211,41.161014],[-8.621019,41.161059],[-8..."
1516431,18,20000455,"[[-8.619822,41.148036],[-8.619831,41.148045],[..."
635989,28,20000632,"[[-8.584443,41.162913],[-8.584425,41.162895],[..."
120741,18,20000142,"[[-8.619795,41.148036],[-8.619804,41.148027],[..."
726124,6,20000421,"[[-8.582517,41.180481],[-8.582193,41.180247],[..."


In [21]:
import json
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# CHECK 1: Does each TAXI_ID always use the same ORIGIN_STAND?
# -------------------------------------------------------------------------
stand_per_taxi = (
    df_clean[df_clean['CALL_TYPE'] == 'B']
    .groupby('TAXI_ID')['ORIGIN_STAND']
    .nunique()
)

print("=== CHECK 1: TAXI_ID vs. ORIGIN_STAND ===")
print(f"Total taxis serving type 'B' trips: {len(stand_per_taxi)}")
print(f"Taxis that use ONLY 1 stand:       {(stand_per_taxi == 1).sum()}")
print(f"Taxis that use multiple (>1) stands: {(stand_per_taxi > 1).sum()}")
print(f"Max stands used by a single taxi:  {stand_per_taxi.max()}\n")


# -------------------------------------------------------------------------
# CHECK 2: Does the first POLYLINE coordinate stay the same for each stand?
# -------------------------------------------------------------------------

# Helper to safely extract start coordinate [lon, lat]
def extract_start_coord(polyline):
    if pd.isna(polyline) or polyline in ('[]', ''):
        return None, None
    try:
        pts = json.loads(polyline) if isinstance(polyline, str) else polyline
        if pts and len(pts) > 0:
            return float(pts[0][0]), float(pts[0][1])
    except Exception:
        pass
    return None, None

# Extract starting longitude and latitude
#coords = df_clean['POLYLINE'].apply(extract_start_coord)
coords = df_clean['POLYLINE'].apply(extract_start_coord).tolist()

# Filling the gaps when missing longitude or latitude (with previous or following coordinates)
for i in range(len(coords)):
    current = coords[i]
    lon, lat = current
    if pd.isna(lon) or pd.isna(lat):
        if 0 <= i - 1 < len(coords) and coords[i - 1] is not None:
            prev_lon, prev_lat = coords[i - 1]
            if not (pd.isna(prev_lon) or pd.isna(prev_lat)):
                coords[i] = (prev_lon, prev_lat)
                print(i)
        elif 0 <= i + 1 < len(coords) and coords[i + 1] is not None:
            next_lon, next_lat = coords[i + 1]
            if not (pd.isna(next_lon) or pd.isna(next_lat)):
                coords[i] = (next_lon, next_lat)
                print(i)

df_clean['START_LON'] = [c[0] for c in coords]
df_clean['START_LAT'] = [c[1] for c in coords]

# Getting rid of rows where starting point has an invalidate longitude or latitude
df_clean = df_clean[~(df_clean['START_LON'].isna() | df_clean['START_LAT'].isna())]

# Analyze coordinate clustering per stand for valid CALL_TYPE == 'B'
stand_geo_stats = (
    df_clean[(df_clean['CALL_TYPE'] == 'B') & df_clean['ORIGIN_STAND'].notna()]
    .groupby('ORIGIN_STAND', observed=False)
    .agg(
        TRIP_COUNT=('TRIP_ID', 'count'),
        UNIQUE_COORDS=('START_LON', lambda x: len(set(zip(x, df_clean.loc[x.index, 'START_LAT'])))),
        LON_MEAN=('START_LON', 'mean'),
        LAT_MEAN=('START_LAT', 'mean'),
        LON_STD=('START_LON', 'std'),
        LAT_STD=('START_LAT', 'std')
    )
)

print("=== CHECK 2: STARTING COORDINATES PER STAND ===")
print(stand_geo_stats.head(10))

=== CHECK 1: TAXI_ID vs. ORIGIN_STAND ===
Total taxis serving type 'B' trips: 439
Taxis that use ONLY 1 stand:       1
Taxis that use multiple (>1) stands: 438
Max stands used by a single taxi:  61

870
886
1026
1190
1876
2088
2129
2909
3092
3182
3191
3427
3433
3436
3441
3468
4027
4057
4467
4597
4661
5095
5509
5907
5919
6522
6778
7235
7401
7552
7671
7675
7706
7882
7906
7979
8600
8723
9216
9313
10042
10226
10230
10487
10655
10770
10873
10891
11255
11267
11540
11961
12134
12185
12287
12290
12380
12390
12392
12444
12669
12709
12725
12738
12744
15013
15024
15111
15121
15552
15713
16997
17483
17563
17952
18001
18495
18506
19776
20060
20163
20228
20572
20635
20819
20828
20849
20859
20860
20882
20945
21304
21697
21832
21833
21897
22080
22341
22433
22479
22635
22672
22689
22734
22754
22926
23136
23317
23319
23479
23483
23505
23548
24140
24788
26025
26385
26641
26764
27301
28266
28278
28379
29501
30427
30589
30712
31318
31392
32135
32215
32460
32860
33196
33307
33616
33624
33747
33940
33994
341

In [23]:
import numpy as np
import pandas as pd
from scipy.spatial import KDTree

# 1. Filter valid stands with non-null start coordinates
valid_b_trips = df_clean[
    (df_clean['CALL_TYPE'] == 'B') & 
    (df_clean['ORIGIN_STAND'].notna()) & 
    (df_clean['START_LON'].notna()) & 
    (df_clean['START_LAT'].notna())
]

# 2. Compute stand centroids (mean start position for each stand)
stand_centroids = (
    valid_b_trips
    .groupby('ORIGIN_STAND', observed=False)[['START_LON', 'START_LAT']]
    .mean()
    .reset_index()
)

# 3. Build spatial KDTree using reference stand locations
tree = KDTree(stand_centroids[['START_LON', 'START_LAT']].values)

# 4. Identify missing ORIGIN_STAND indices for CALL_TYPE 'B'
missing_mask = (
    (df_clean['CALL_TYPE'] == 'B') & 
    (df_clean['ORIGIN_STAND'].isna()) & 
    (df_clean['START_LON'].notna()) & 
    (df_clean['START_LAT'].notna())
)

missing_coords = df_clean.loc[missing_mask, ['START_LON', 'START_LAT']].values

# 5. Query nearest stand ID for each missing point
if len(missing_coords) > 0:
    distances, indices = tree.query(missing_coords)
    assigned_stands = stand_centroids.iloc[indices]['ORIGIN_STAND'].values
    
    # Fill missing values
    df_clean.loc[missing_mask, 'ORIGIN_STAND'] = assigned_stands

# 6. Verify final completeness
missing_remaining = df_clean[df_clean['CALL_TYPE'] == 'B']['ORIGIN_STAND'].isna().sum()
print(f"Remaining missing ORIGIN_STAND for CALL_TYPE 'B': {missing_remaining}")

Remaining missing ORIGIN_STAND for CALL_TYPE 'B': 0


In [24]:
import json
import numpy as np

from haversine import haversine_vector, Unit


def max_speed_kmh(points, interval_seconds=15):
    if len(points) < 2:
        return 0

    # POLYLINE stores [longitude, latitude]
    # haversine expects [latitude, longitude]
    points = np.asarray(points)

    coordinates = np.column_stack((
        points[:, 1],  # latitude
        points[:, 0]   # longitude
    ))

    point1 = coordinates[:-1]
    point2 = coordinates[1:]

    # Use vector function to hopefully save time. (This still took like 15 min for me at least)
    distances_km = haversine_vector(
        point1,
        point2,
        Unit.KILOMETERS
    )

    # Convert distance per 15 seconds to km/h
    speed_kmh = distances_km / (interval_seconds / 3600)

    return np.max(speed_kmh)


# Convert POLYLINE from JSON strings to Python lists
# and calculate maximum speed for every trip.
df_clean["MAX_SPEED_KMH"] = (
    df_clean["POLYLINE"]
    .apply(json.loads)
    .apply(max_speed_kmh)
)


# Find trips where at least one segment exceeds 300 km/h
invalid_trips = df_clean[
    df_clean["MAX_SPEED_KMH"] > 300
]

print(f"Invalid trips: {len(invalid_trips)}")

Invalid trips: 14948


In [25]:

import json
import haversine as hv

print(df_clean["MAX_SPEED_KMH"])
print(df_clean["POLYLINE"].iloc[2])

# Basically copied from cell above. Only used to double check the speed calculation
def print_speeds(points, interval_seconds=15):
    if len(points) < 2:
        return 0

    max_speed = 0

    for p1, p2 in zip(points[:-1], points[1:]):

        # POLYLINE stores coordinates as [longitude, latitude]
        # haversine expects (latitude, longitude)
        point1 = (p1[1], p1[0])
        point2 = (p2[1], p2[0])

        distance_km = hv.haversine(
            point1,
            point2,
            unit=hv.Unit.KILOMETERS
        )

        # Each POLYLINE point represents 15 seconds. Assumes consistent data (from cleaning)
        speed_kmh = distance_km / (interval_seconds / 3600)

        print(
            f"Speed: {speed_kmh:.2f} km/h | "
            f"Points: {point1} -> {point2}"
        )

        max_speed = max(max_speed, speed_kmh)

    return max_speed


# Test the third trajectory
points = json.loads(df_clean["POLYLINE"].iloc[2])

max_speed = print_speeds(points)

print(f"\nMaximum speed: {max_speed:.2f} km/h")

577         68.436106
3          532.166091
0           68.169886
10          57.414548
64         427.804925
              ...    
1710412     27.180759
1710511     17.205180
1710603     56.157033
1710532      1.766618
1710310      0.000000
Name: MAX_SPEED_KMH, Length: 1710589, dtype: float64
[[-8.618643,41.141412],[-8.618499,41.141376],[-8.620326,41.14251],[-8.622153,41.143815],[-8.623953,41.144373],[-8.62668,41.144778],[-8.627373,41.144697],[-8.630226,41.14521],[-8.632746,41.14692],[-8.631738,41.148225],[-8.629938,41.150385],[-8.62911,41.151213],[-8.629128,41.15124],[-8.628786,41.152203],[-8.628687,41.152374],[-8.628759,41.152518],[-8.630838,41.15268],[-8.632323,41.153022],[-8.631144,41.154489],[-8.630829,41.154507],[-8.630829,41.154516],[-8.630829,41.154498],[-8.630838,41.154489]]
Speed: 3.05 km/h | Points: (41.141412, -8.618643) -> (41.141376, -8.618499)
Speed: 47.58 km/h | Points: (41.141376, -8.618499) -> (41.14251, -8.620326)
Speed: 50.61 km/h | Points: (41.14251, -8.620326) ->

In [26]:
df_clean = df_clean[df_clean["MAX_SPEED_KMH"] <= 300]

In [27]:
import folium
from folium.plugins import MarkerCluster

# 1. Calculate stand centroids and total trip volume
stand_centroids = (
    df_clean[
        (df_clean['CALL_TYPE'] == 'B') & 
        (df_clean['ORIGIN_STAND'].notna()) & 
        (df_clean['START_LON'].notna()) & 
        (df_clean['START_LAT'].notna())
    ]
    .groupby('ORIGIN_STAND', observed=False)
    .agg(
        LON=('START_LON', 'mean'),
        LAT=('START_LAT', 'mean'),
        TRIP_COUNT=('TRIP_ID', 'count')
    )
    .reset_index()
)

# 2. Create base map centered on Porto, Portugal
porto_map = folium.Map(
    location=[41.1501, -8.6103], 
    zoom_start=13, 
    tiles='OpenStreetMap'
)

# 3. Add clustered or circle markers for each stand
marker_cluster = MarkerCluster().add_to(porto_map)

for _, row in stand_centroids.iterrows():
    stand_id = int(row['ORIGIN_STAND'])
    trips = int(row['TRIP_COUNT'])
    lat, lon = row['LAT'], row['LON']
    
    folium.CircleMarker(
        location=[lat, lon],
        radius=min(15, max(4, trips // 2000)),  # Size scaled by popularity
        popup=f"<b>Taxi Stand {stand_id}</b><br>Total Trips: {trips:,}",
        color='#2b8cbe',
        fill=True,
        fill_color='#7bccc4',
        fill_opacity=0.7
    ).add_to(marker_cluster)

# Display in Jupyter notebook
porto_map

In [28]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Set plotting style
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------------------
# 1. CALL_TYPE Distribution
# -------------------------------------------------------------------------
call_counts = df_clean['CALL_TYPE'].value_counts()
total_trips = len(df_clean)

sns.barplot(x=call_counts.index, y=call_counts.values, ax=axes[0, 0], palette='Blues_r')
axes[0, 0].set_title('CALL_TYPE Distribution', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Call Type (A = Central, B = Stand, C = Street)')
axes[0, 0].set_ylabel('Number of Trips')

for i, count in enumerate(call_counts.values):
    pct = (count / total_trips) * 100
    axes[0, 0].text(i, count + (total_trips * 0.01), f"{count:,}\n({pct:.1f}%)", 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

# -------------------------------------------------------------------------
# 2. DAY_TYPE Distribution
# -------------------------------------------------------------------------
day_col = 'DAY_TYPE' if 'DAY_TYPE' in df_clean.columns else 'DAYTYPE'
day_counts = df_clean[day_col].value_counts()

sns.barplot(x=day_counts.index, y=day_counts.values, ax=axes[0, 1], palette='Greens_r')
axes[0, 1].set_title('DAY_TYPE Distribution', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Day Type (A = Normal, B = Holiday, C = Pre-Holiday)')
axes[0, 1].set_ylabel('Number of Trips')

for i, count in enumerate(day_counts.values):
    pct = (count / total_trips) * 100
    axes[0, 1].text(i, count + (total_trips * 0.01), f"{count:,}\n({pct:.1f}%)", 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

# -------------------------------------------------------------------------
# 3. Top 3 and Bottom 3 Taxis by Number of Rides
# -------------------------------------------------------------------------
taxi_counts = df_clean['TAXI_ID'].value_counts()
top_3_taxis = taxi_counts.head(3)
bottom_3_taxis = taxi_counts.tail(3)

# Combine Top 3 and Bottom 3 into a single DataFrame for plotting
top_bottom_df = pd.concat([
    pd.DataFrame({'TAXI_ID': top_3_taxis.index.astype(str), 'RIDE_COUNT': top_3_taxis.values, 'GROUP': 'Top 3'}),
    pd.DataFrame({'TAXI_ID': bottom_3_taxis.index.astype(str), 'RIDE_COUNT': bottom_3_taxis.values, 'GROUP': 'Bottom 3'})
])

colors = ['#2ca02c' if g == 'Top 3' else '#d62728' for g in top_bottom_df['GROUP']]

sns.barplot(data=top_bottom_df, x='TAXI_ID', y='RIDE_COUNT', ax=axes[1, 0], palette=colors)
axes[1, 0].set_title('Top 3 vs. Bottom 3 Taxis by Ride Volume', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('TAXI_ID')
axes[1, 0].set_ylabel('Total Rides')
axes[1, 0].tick_params(axis='x', rotation=30)

for i, row in enumerate(top_bottom_df.itertuples()):
    axes[1, 0].text(i, row.RIDE_COUNT + 10, f"{row.RIDE_COUNT:,}", 
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

# -------------------------------------------------------------------------
# 4. Stand-Alone Text Summary Card in the 4th Slot
# -------------------------------------------------------------------------
axes[1, 1].axis('off')

top_str = "\n".join([f"  • Taxi {tid}: {cnt:,} rides" for tid, cnt in top_3_taxis.items()])
bot_str = "\n".join([f"  • Taxi {tid}: {cnt:,} rides" for tid, cnt in bottom_3_taxis.items()])

summary_text = (
    f"=== TAXI RIDE EXTREMES ===\n\n"
    f"MOST ACTIVE TAXIS:\n{top_str}\n\n"
    f"LEAST ACTIVE TAXIS:\n{bot_str}\n\n"
    f"Total Active Taxis: {df_clean['TAXI_ID'].nunique():,}"
)

axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace', va='center')

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [7]:
from DbConnector import DbConnector
from tabulate import tabulate
import mysql.connector as mysql
import haversine as hv

class DbConnector:
    """
    Connects to the MySQL server on the Ubuntu virtual machine.
    """

    def __init__(self,
                 HOST="tdt4225-03.idi.ntnu.no",
                 DATABASE="db_A2_G3",
                 USER="DiegoT",
                 PASSWORD="G3Rocky"):
        self.db_connection = None
        self.cursor = None

        try:
            self.db_connection = mysql.connect(
                host=HOST,
                database=DATABASE,
                user=USER,
                password=PASSWORD,
                port=3306
            )
            self.cursor = self.db_connection.cursor()

            print("Connected to:", self.db_connection.get_server_info())
            self.cursor.execute("SELECT DATABASE();")
            database_name = self.cursor.fetchone()
            print("You are connected to the database:", database_name)
            print("-----------------------------------------------\n")

        except mysql.Error as e:
            print("ERROR: Failed to connect to db:", e)
            raise e  # Re-raise so main() handles setup failure cleanly

    def close_connection(self):
        if self.cursor:
            self.cursor.close()
        if self.db_connection and self.db_connection.is_connected():
            print("\n-----------------------------------------------")
            print("Connection to %s is closed" % self.db_connection.get_server_info())
            self.db_connection.close()

class Program:
    def __init__(self):
        self.connection = DbConnector()
        self.db_connection = self.connection.db_connection
        self.cursor = self.connection.cursor

    def create_table(self, query, table_name):
        # This adds table_name to the %s variable and executes the query
        self.cursor.execute(query % table_name)
        self.db_connection.commit()

    def insert_data(self, data):
        trips_insert_query = """ INSERT INTO trips (
                                    trip_id,
                                    taxi_id,
                                    call_type,
                                    origin_call,
                                    origin_stand,
                                    start_time,
                                    missing_data
                                    )
                                    VALUES (%s, %s, %s, %s, %s, %s, %s)
                            """
        points_insert_query = """ INSERT INTO points (
                                    trip_id,
                                    point_idx,
                                    latitude,
                                    longitude
                                    )
                                    VALUES (%s, %s, %s, %s)
                            """
        
        for _, row in data.iterrows():
            # Had an issue with Null values
            origin_call = (
                None if pd.isna(row["ORIGIN_CALL"])
                else row["ORIGIN_CALL"]
            )
            origin_stand = (
                None if pd.isna(row["ORIGIN_STAND"])
                else row["ORIGIN_STAND"]
            )

            start_time = row["START_TIME"].to_pydatetime()
            
            # Insert trips
            trip_vals = (
                row["TRIP_ID"],
                row["TAXI_ID"],
                row["CALL_TYPE"],
                origin_call,
                origin_stand,
                start_time,
                row["MISSING_DATA"]
            )
            self.cursor.execute(trips_insert_query, trip_vals)

            # Insert coords
            points = json.loads(row["POLYLINE"])
            for i, point in enumerate(points):
                point_val = (
                    row["TRIP_ID"],
                    i,
                    point[1], # latitude (i hope)
                    point[0]
                )
                self.cursor.execute(points_insert_query, point_val)
                   
        self.db_connection.commit()

    def insert_data_fast(self, data):

        trips_insert_query = """
            INSERT INTO trips (
                trip_id,
                taxi_id,
                call_type,
                origin_call,
                origin_stand,
                start_time,
                missing_data
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """

        points_insert_query = """
            INSERT INTO points (
                trip_id,
                point_idx,
                latitude,
                longitude
            )
            VALUES (%s, %s, %s, %s)
        """

        # Removing dupes (MAYBE CHANGE THIS)
        data = data.drop_duplicates(
            subset=["TRIP_ID"],
            keep="first"
        )

        batch_size = 5000

        trip_values = []
        point_values = []

        total_rows = len(data)

        for count, (_, row) in enumerate(data.iterrows(), start=1):

            # Convert missing values to Python None
            origin_call = (
                None
                if pd.isna(row["ORIGIN_CALL"])
                else int(row["ORIGIN_CALL"])
            )

            origin_stand = (
                None
                if pd.isna(row["ORIGIN_STAND"])
                else int(row["ORIGIN_STAND"])
            )

            # Convert pandas Timestamp to Python datetime
            start_time = row["START_TIME"].to_pydatetime()

            # Add trip to batch
            trip_values.append((
                int(row["TRIP_ID"]),
                int(row["TAXI_ID"]),
                row["CALL_TYPE"],
                origin_call,
                origin_stand,
                start_time,
                bool(row["MISSING_DATA"])
            ))

            # Add all GPS points for this trip to batch
            points = json.loads(row["POLYLINE"])

            for i, point in enumerate(points):
                point_values.append((
                    int(row["TRIP_ID"]),
                    i,
                    point[1],  # latitude
                    point[0]   # longitude
                ))

            # Insert every 5000 trips
            if count % batch_size == 0 or count == total_rows:

                print(
                    f"Inserting {count:,}/{total_rows:,} trips "
                    f"({count / total_rows * 100:.1f}%)"
                )

                # Insert trips
                self.cursor.executemany(
                    trips_insert_query,
                    trip_values
                )

                # Insert GPS points
                if point_values:
                    self.cursor.executemany(
                        points_insert_query,
                        point_values
                    )

                # Commit this batch
                self.db_connection.commit()

                # Clear batches
                trip_values.clear()
                point_values.clear()

        print("Finished inserting data.")

    def fetch_data(self, table_name):
        query = "SELECT * FROM %s"
        self.cursor.execute(query % table_name)
        rows = self.cursor.fetchall()
        print("Data from table %s, raw format:" % table_name)
        print(rows)
        # Using tabulate to show the table in a nice way
        print("Data from table %s, tabulated:" % table_name)
        print(tabulate(rows, headers=self.cursor.column_names))
        return rows

    def drop_table(self, table_name):
        print("Dropping table %s..." % table_name)
        query = "DROP TABLE IF EXISTS %s"
        self.cursor.execute(query % table_name)

    def show_tables(self):
        self.cursor.execute("SHOW TABLES")
        rows = self.cursor.fetchall()
        print(tabulate(rows, headers=self.cursor.column_names))

    def initiate_db(self, data):
        self.drop_table("points")
        self.drop_table("trips")
        
        trips_query = """CREATE TABLE IF NOT EXISTS %s (
                            trip_id BIGINT NOT NULL PRIMARY KEY,
                            taxi_id BIGINT NOT NULL,
                            call_type CHAR(1),
                            origin_call BIGINT NULL,
                            origin_stand INT NULL,
                            start_time DATETIME NOT NULL,
                            missing_data BOOLEAN NOT NULL
                            )
                        """
        self.create_table(trips_query, "trips")

        points_query = """CREATE TABLE IF NOT EXISTS %s (
                            trip_id BIGINT NOT NULL,
                            point_idx INT NOT NULL,
                            latitude DECIMAL(10, 7) NOT NULL,
                            longitude DECIMAL(10, 7) NOT NULL,

                            PRIMARY KEY (trip_id, point_idx),

                            FOREIGN KEY (trip_id)
                                REFERENCES trips(trip_id)
                                ON DELETE CASCADE
                            )
                        """
        self.create_table(points_query, "points")
        self.cursor.execute("DESCRIBE points")
        rows = self.cursor.fetchall()
        print(tabulate(rows, headers=self.cursor.column_names))
        self.insert_data_fast(data)

    def check_data(self):
        self.cursor.execute("SELECT * FROM trips LIMIT 5")
        print(tabulate(self.cursor.fetchall(), headers=self.cursor.column_names))
        
        self.cursor.execute("SELECT * FROM points LIMIT 10")
        print(tabulate(self.cursor.fetchall(), headers=self.cursor.column_names))

        self.cursor.execute("SELECT COUNT(*) FROM trips")
        print(self.cursor.fetchall())

        # Takes a while
        #self.cursor.execute("SELECT COUNT(*) FROM points")
        #print(self.cursor.fetchall())

    def exercise_1(self):
        queries = [
            ("""
                SELECT
                    COUNT(DISTINCT taxi_id) AS nb_taxis
                FROM trips
            """),
            ("""
                SELECT
                    COUNT(DISTINCT trip_id) AS nb_trips
                FROM trips
            """),
            ("""
                SELECT
                    COUNT(trip_id) AS nb_points
                FROM points
            """)
        ]
 
        for query in queries:
            self.cursor.execute(query)
            results = self.cursor.fetchall()
            print(tabulate(results[:10], headers=self.cursor.column_names))

    def exercise_2(self):
        query = """
                    SELECT
                        AVG(nb_trips_per_taxi) AS avg_trips_per_taxi
                    FROM (
                        SELECT
                            COUNT(DISTINCT trip_id) as nb_trips_per_taxi
                        FROM trips
                        GROUP BY taxi_id
                    ) AS t;
                """
 
        self.cursor.execute(query)
        results = self.cursor.fetchall()
        print(tabulate(results[:10], headers=self.cursor.column_names))
 
    def exercise_3(self):
        query = """
                    SELECT
                        taxi_id,
                        COUNT(DISTINCT trip_id) as nb_trips_per_taxi
                    FROM trips
                    GROUP BY taxi_id
                    ORDER BY nb_trips_per_taxi DESC
                    LIMIT 20
                """
 
        self.cursor.execute(query)
        results = self.cursor.fetchall()
        print(tabulate(results[:10], headers=self.cursor.column_names))
 
    def exercise_4_a(self):
        query = """
                    SELECT
                        taxi_id,
                        CASE
                            WHEN nb_calls_type_A >= nb_calls_type_B AND nb_calls_type_A >= nb_calls_type_C THEN
                                'A'
                            WHEN nb_calls_type_B >= nb_calls_type_A AND nb_calls_type_B >= nb_calls_type_C THEN
                                'B'
                            ELSE 'C'
                        END AS most_used_call_type
                    FROM(
                        SELECT
                            taxi_id,
                            SUM(CASE WHEN call_type='A' THEN 1 ELSE 0 END) AS nb_calls_type_A,
                            SUM(CASE WHEN call_type='B' THEN 1 ELSE 0 END) AS nb_calls_type_B,
                            SUM(CASE WHEN call_type='C' THEN 1 ELSE 0 END) AS nb_calls_type_C
                        FROM trips
                        GROUP BY taxi_id
                    ) AS t
                """
 
        self.cursor.execute(query)
        results = self.cursor.fetchall()
        print(tabulate(results[:10], headers=self.cursor.column_names))

    def exercise_4_b(self):
        query1 = """
                    SELECT
                        t.trip_id,
                        t.call_type,
                        JSON_ARRAYAGG(
                            JSON_ARRAY(p.longitude, p.latitude)
                        ) AS list_points
                    FROM trips t
                    JOIN points p
                        ON p.trip_id = t.trip_id
                    GROUP BY t.trip_id, t.call_type
                    ORDER BY t.trip_id
                """
 
        self.cursor.execute(query1)
        data = self.cursor.fetchall()
        durations = {'A': [], 'B': [], 'C': []}
        distances = {'A': [], 'B': [], 'C': []}
        for row in data:
            type = row[1]
            points = json.loads(row[2])
            durations[type].append(len(points)*15)    # The number of points multiplied by 15 gives the duration of the trip in seconds since the coordinates are sampled every 15 seconds
            distance = 0
            for i in range(len(points)-1):
                distance += hv.haversine(points[i], points[i+1], unit="m")
            distances[type].append(distance)
 
        for key, value in durations.items():
            print(f"Mean duration of trips of type {key}: {round(np.mean(value), 2)} seconds")
 
        print("\n")
 
        for key, value in distances.items():
            print(f"Mean distance of trips of type {key}: {round(np.mean(value), 2)} meters")
 
        print("\n")
 
        query2 = """
                    SELECT
                        CASE
                            WHEN HOUR(start_time) >= 0 AND HOUR(start_time) < 6 THEN '00-06'
                            WHEN HOUR(start_time) >= 6 AND HOUR(start_time) < 12 THEN '06-12'
                            WHEN HOUR(start_time) >= 12 AND HOUR(start_time) < 18 THEN '12-18'
                            ELSE '18-24'
                        END AS time_band,
                    COUNT(*) AS trip_count
                    FROM trips
                    GROUP BY
                        CASE
                            WHEN HOUR(start_time) >= 0 AND HOUR(start_time) < 6 THEN '00-06'
                            WHEN HOUR(start_time) >= 6 AND HOUR(start_time) < 12 THEN '06-12'
                            WHEN HOUR(start_time) >= 12 AND HOUR(start_time) < 18 THEN '12-18'
                            ELSE '18-24'
                        END
                """
        
        self.cursor.execute(query2)
        data = self.cursor.fetchall()
        for time_band, nb_trips in data:
            print(f"Number of trips during time band {time_band}: {nb_trips}")

    def exercise_8(self):
        query = """
                    SELECT
                        t.trip_id,
                        t.start_time,
                        DATE_ADD(
                            t.start_time,
                            INTERVAL last_point.point_idx * 15 SECOND
                        ) AS estimated_end_time
                    FROM trips t

                    JOIN (
                        SELECT
                            trip_id,
                            point_idx
                        FROM points p
                        WHERE NOT EXISTS (
                            SELECT 1
                            FROM points p2
                            WHERE p2.trip_id = p.trip_id
                            AND p2.point_idx > p.point_idx
                        )
                    ) AS last_point
                        ON t.trip_id = last_point.trip_id

                    WHERE t.missing_data = 0
                    AND DATE(t.start_time) <
                        DATE(
                            DATE_ADD(
                                t.start_time,
                                INTERVAL last_point.point_idx * 15 SECOND
                            )
                        )

                    ORDER BY t.start_time
                """
        self.cursor.execute(query)
        midnight_crossers = self.cursor.fetchall()
        print(tabulate(midnight_crossers[:10], headers=self.cursor.column_names))

    def exercise_9(self):
        query = """
                    SELECT
                        t.trip_id,
                        start_point.latitude AS start_latitude,
                        start_point.longitude AS start_longitude,
                        end_point.latitude AS end_latitude,
                        end_point.longitude AS end_longitude
                    FROM trips t

                    JOIN points start_point
                        ON start_point.trip_id = t.trip_id
                        AND start_point.point_idx = 0

                    JOIN points end_point
                        ON end_point.trip_id = t.trip_id

                    WHERE t.missing_data = 0

                    AND NOT EXISTS (
                        SELECT 1
                        FROM points later_point
                        WHERE later_point.trip_id = end_point.trip_id
                            AND later_point.point_idx > end_point.point_idx
                    )
                """

        self.cursor.execute(query)
        data = self.cursor.fetchall()
        circular_trips = []
        for row in data:
            start_point = (row[1], row[2])
            end_point = (row[3], row[4])

            distance = hv.haversine(start_point, end_point, unit="m")

            if distance <= 50:
                circular_trips.append((
                    row[0],
                    row[1],
                    row[2],
                    row[3],
                    row[4],
                    distance
                ))

        print(f"Circual trip count: {len(circular_trips)}")
        
        print(
            tabulate(
                circular_trips[:20],
                headers=[
                    "trip_id",
                    "start_latitude",
                    "start_longitude",
                    "end_latitude",
                    "end_longitude",
                    "distance_m"
                ]
            )
        )

    def exercise_10(self):
        query = """
                    WITH trip_times AS (
                        SELECT
                            t.trip_id,
                            t.taxi_id,
                            t.start_time,
                            DATE_ADD(
                                t.start_time,
                                INTERVAL (
                                    SELECT MAX(p.point_idx) * 15
                                    FROM points p
                                    WHERE p.trip_id = t.trip_id
                                ) SECOND
                            ) AS end_time
                        FROM trips t
                        WHERE t.missing_data = 0
                    ),

                    consecutive_trips AS (
                        SELECT
                            taxi_id,
                            trip_id,
                            start_time,
                            end_time,
                            LAG(end_time) OVER (
                                PARTITION BY taxi_id
                                ORDER BY start_time
                            ) AS previous_end_time
                        FROM trip_times
                    )

                    SELECT
                        taxi_id,
                        AVG(
                            TIMESTAMPDIFF(
                                SECOND,
                                previous_end_time,
                                start_time
                            )
                        ) / 60 AS average_idle_minutes
                    FROM consecutive_trips
                    WHERE previous_end_time IS NOT NULL
                    AND start_time >= previous_end_time
                    GROUP BY taxi_id
                    ORDER BY average_idle_minutes DESC
                    LIMIT 20
                """

        self.cursor.execute(query)

        rows = self.cursor.fetchall()

        print(
            tabulate(
                rows,
                headers=self.cursor.column_names
            )
        )


def main():
    program = None
    try:
        program = Program()

        #program.initiate_db(df_clean)

        #program.check_data()

        program.exercise_9()

        # Check that the table is dropped
        program.show_tables()
    except Exception as e:
        print("ERROR: Failed to use database:", e)
    finally:
        if program:
            program.connection.close_connection()


if __name__ == '__main__':
    main()
    print("Done")

Connected to: 8.4.11-0ubuntu0.26.04.1
You are connected to the database: ('db_A2_G3',)
-----------------------------------------------

Circual trip count: 55859
            trip_id    start_latitude    start_longitude    end_latitude    end_longitude    distance_m
-------------------  ----------------  -----------------  --------------  ---------------  ------------
1372638303620000112           41.1624           -8.58712         41.1624         -8.58714       2.47189
1372639092620000233           41.1683           -8.63274         41.1682         -8.6328        8.76831
1372642886620000403           41.1458           -8.61091         41.1458         -8.61091       0
1372644011620000463           41.1464           -8.61766         41.1464         -8.61766       0
1372644642620000305           41.1445           -8.60593         41.1447         -8.60632      40.944
1372644692620000173           41.1456           -8.59819         41.1456         -8.59819       0
1372650711620000403       